# Added in this excercise
- Storing unanswered questions and user details in database
- database tables can be listed


In [ ]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr
import sqlite3


In [ ]:
load_dotenv(override=True)
openai = OpenAI()
DB_NAME = "vector_db"
MODEL ="gpt-4o-mini"

In [ ]:
DB_PATH = "excercise.db"

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
    CREATE TABLE IF NOT EXISTS user_detail (
        email TEXT,
        name TEXT,
        notes TEXT
    )
    """)
    conn.execute("""
    CREATE TABLE IF NOT EXISTS unknown_question (
        question TEXT,
        timestamp TEXT
    )
    """)

print(f"SQLite database initialized at {DB_PATH}")

In [ ]:
def get_user_details(email):
    with sqlite3.connect(DB_PATH) as conn:
        row = conn.execute(
            "SELECT email, name, notes FROM user_detail WHERE email = ?",
            (email,),
        ).fetchone()
    if not row:
        return None
    return {"email": row[0], "name": row[1], "notes": row[2]}

In [ ]:
def get_table(table_name):
    with sqlite3.connect(DB_PATH) as conn:
        table_exists = conn.execute(
            "SELECT name FROM sqlite_master WHERE type = 'table' AND name = ?",
            (table_name,),
        ).fetchone()
        if not table_exists:
            return f"Table '{table_name}' does not exist."
        cursor = conn.execute(f'SELECT * FROM "{table_name}"')
        columns = [description[0] for description in cursor.description]
        rows = cursor.fetchall()

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = [
        "| " + " | ".join(str(value).replace("|", "\\|") for value in row) + " |"
        for row in rows
    ]
    if not body:
        body = ["| " + " | ".join([""] * len(columns)) + " |"]
    return "\n".join([header, separator, *body])

In [ ]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    existing_user = get_user_details(email)
    if existing_user:
        print(f"User with email {email} already exists")
        return {"recorded": "already exists"}
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO user_detail (email, name, notes) VALUES (?, ?, ?)",
            (email, name, notes),
        )
    print(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [ ]:
def record_unknown_question(question):
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute(
            "INSERT INTO unknown_question (question, timestamp) VALUES (?, CURRENT_TIMESTAMP)",
            (question,),
        )
    print(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
get_table_json = {
    "name": "get_table",
    "description": "List a content of a sqlite table in markdown format, eg. user_detail, unknown_question",
    "parameters": {
        "type": "object",
        "properties": {
            "table_name": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["table_name"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
    {"type": "function", "function": get_table_json},
        ]

In [ ]:
# This is a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
reader = PdfReader("../../me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("../../me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Ed Donner"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # This is the call to the LLM - see that we pass in the tools json

        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason

        # If the LLM wants to call a tool, we do that!

        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat, type="messages").launch()